In [ ]:
import random
import numpy as np
from utils.dataset import ClaudetteDataset
from utils.llm import get_llm_task_response
from utils.prompts import BINARY_TASK_INSTRUCTION
from utils.metrics import compute_binary_metrics, display_metrics

STRATEGIES = {
    "cot": "Let's think step-by-step before answering.",
    "emotion": "Consider the emotional tone of the sentence.",
    "rephrase": "Rephrase the sentence before judging."
}

META_PROMPT_TEMPLATE = """{strategy}
Classify whether the following clause is UNFAIR or FAIR: \"{sentence}\""""

cd = ClaudetteDataset()
train_df = cd.get_dataset('train')
DATA = list(zip(train_df['text'], train_df['label']))

def llm_classify(clause: str) -> str:
    resp = get_llm_task_response(user_prompt=clause, system_prompt=BINARY_TASK_INSTRUCTION)
    return resp.content.strip().upper()

def evaluate_prompt(prompt: str, label: int) -> int:
    pred = llm_classify(prompt)
    pred_label = 1 if 'UNFAIR' in pred else 0
    return int(pred_label == label)

class ThompsonSampler:
    def __init__(self, strategy_names):
        self.strategies = strategy_names + ['none']
        self.alpha = {s: 1 for s in self.strategies}
        self.beta = {s: 1 for s in self.strategies}

    def sample(self):
        samples = {s: np.random.beta(self.alpha[s], self.beta[s]) for s in self.strategies}
        return max(samples, key=samples.get)

    def update(self, strategy, reward):
        if reward:
            self.alpha[strategy] += 1
        else:
            self.beta[strategy] += 1

# Add a global variable to store the winning strategy
WINNING_STRATEGY = None

def run_opts_ts(data, strategies, rounds=10):
    global WINNING_STRATEGY
    ts = ThompsonSampler(list(strategies.keys()))
    strategy_rewards = {s: 0 for s in ts.strategies}
    strategy_counts = {s: 0 for s in ts.strategies}
    for i in range(rounds):
        sentence, label = random.choice(data)
        strategy = ts.sample()
        if strategy == 'none':
            prompt = f"Classify whether the following clause is UNFAIR or FAIR: \"{sentence}\""
        else:
            prompt = META_PROMPT_TEMPLATE.format(strategy=strategies[strategy], sentence=sentence)
        reward = evaluate_prompt(prompt, label)
        ts.update(strategy, reward)
        strategy_rewards[strategy] += reward
        strategy_counts[strategy] += 1
        print(f"Round {i+1:02d}: Strategy={strategy:<8} | Reward={reward}")
    print("\nFinal strategy beliefs (alpha / beta):")
    for s in ts.strategies:
        print(f"{s:<8}: {ts.alpha[s]} / {ts.beta[s]}")
    # Determine the best strategy by average reward
    best_strategy = max(strategy_rewards, key=lambda s: (strategy_rewards[s] / strategy_counts[s]) if strategy_counts[s] > 0 else 0)
    WINNING_STRATEGY = best_strategy
    print(f"\nWinning strategy: {best_strategy}")
    # Save the winning prompt template
    if best_strategy == 'none':
        winning_prompt = "Classify whether the following clause is UNFAIR or FAIR: \"{sentence}\""
    else:
        winning_prompt = META_PROMPT_TEMPLATE.format(strategy=strategies[best_strategy], sentence="{sentence}")
    print(f"Winning prompt template:\n{winning_prompt}")
    # Optionally, save to file
    with open("winning_prompt.txt", "w", encoding="utf-8") as f:
        f.write(winning_prompt)

def run_opts_apet(data, strategies, base_p=0.5, rounds=10):
    for i in range(rounds):
        sentence, label = random.choice(data)
        apply = np.random.choice([True, False], p=[base_p, 1 - base_p])
        if apply:
            strategy = random.choice(list(strategies.keys()))
            prompt = META_PROMPT_TEMPLATE.format(strategy=strategies[strategy], sentence=sentence)
        else:
            strategy = 'none'
            prompt = f"Classify whether the following clause is UNFAIR or FAIR: \"{sentence}\""
        reward = evaluate_prompt(prompt, label)
        print(f"Round {i+1:02d}: Strategy={strategy:<8} | Reward={reward}")

def classify_dataset(df, sample_size=50):
    # Use the winning strategy's prompt
    global WINNING_STRATEGY
    if WINNING_STRATEGY is None:
        # Load from file if not set
        try:
            with open("winning_prompt.txt", "r", encoding="utf-8") as f:
                winning_prompt = f.read()
        except FileNotFoundError:
            print("No winning prompt found. Please run run_opts_ts first.")
            return
    else:
        if WINNING_STRATEGY == 'none':
            winning_prompt = "Classify whether the following clause is UNFAIR or FAIR: \"{sentence}\""
        else:
            winning_prompt = META_PROMPT_TEMPLATE.format(strategy=STRATEGIES[WINNING_STRATEGY], sentence="{sentence}")

    df_s = df.sample(sample_size, random_state=42) if sample_size else df
    preds = []
    for clause in df_s['text']:
        prompt = winning_prompt.format(sentence=clause)
        resp = llm_classify(prompt)
        preds.append(1 if 'UNFAIR' in resp else 0)
    metrics = compute_binary_metrics(df_s['label'], preds)
    display_metrics('LLM classification', metrics)


In [5]:
run_opts_ts(DATA, STRATEGIES, rounds=5)
print()
run_opts_apet(DATA, STRATEGIES, rounds=5)

print("\nEvaluating on a sample of the test set...")
test_df = cd.get_dataset('test')
classify_dataset(test_df, sample_size=20)


Round 01: Strategy=cot      | Reward=1
Round 02: Strategy=emotion  | Reward=0
Round 03: Strategy=cot      | Reward=1
Round 04: Strategy=none     | Reward=1
Round 05: Strategy=cot      | Reward=0

Final strategy beliefs (alpha / beta):
cot     : 3 / 2
emotion : 1 / 2
rephrase: 1 / 1
none    : 2 / 1

Round 01: Strategy=rephrase | Reward=1
Round 02: Strategy=none     | Reward=1
Round 03: Strategy=rephrase | Reward=1
Round 04: Strategy=cot      | Reward=1
Round 05: Strategy=none     | Reward=0

Evaluating on a sample of the test set...
=== LLM classification ===
-- Binary classification --
accuracy: 0.7500
micro_f1: 0.7500
macro_f1: 0.4286
precision: 1.0000
recall: 0.7500

